# DCGAN — Deep Convolutional GAN

This notebook implements **DCGAN** (Radford et al., 2015), following Module 9. The vanilla GAN told us *what* to optimize; DCGAN tells us *how to architect it* for images.

DCGAN's core design principles:

- **No fully connected hidden layers** — preserve spatial structure with convolutions
- **Generator** uses *transposed convolutions* to go from low → high resolution
- **Discriminator** uses *strided convolutions* to go from high → low resolution
- **BatchNorm** in both networks
- **ReLU** in the Generator, **LeakyReLU** in the Discriminator
- **Tanh** Generator output, with data normalized to $[-1, +1]$
- **Weight init** following the DCGAN paper recommendations

We again train on MNIST, but **resize to 64×64** so the architecture dimensions are clean powers of two.

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torchvision.utils import save_image, make_grid
import matplotlib.pyplot as plt
import numpy as np
import os

torch.manual_seed(42)
np.random.seed(42)

## 1. Setup and Hyperparameters

Same DCGAN defaults as before: Adam, `lr=2e-4`, `betas=(0.5, 0.999)`. We bump the image size from 28 to 64 to fit the convolutional architecture cleanly.

In [2]:
latent_dim    = 100
img_channels  = 1    # MNIST grayscale
img_size      = 64   # upscaled from 28 for clean power-of-2 dims
features_g    = 64   # base width of the Generator
features_d    = 64   # base width of the Discriminator
batch_size    = 64
lr            = 2e-4
betas         = (0.5, 0.999)
epochs        = 25

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

os.makedirs('samples', exist_ok=True)

Using device: cpu


## 2. Data — MNIST upscaled to 64×64

MNIST is 28×28. We resize to 64×64 so the architecture can follow the clean $1 \to 4 \to 8 \to 16 \to 32 \to 64$ progression without odd dimensions.

The data is still normalized to $[-1, +1]$ so the Generator's `Tanh` output matches.

In [3]:
transform = transforms.Compose([
    transforms.Resize(img_size),
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5]),    # (0, 1) -> (-1, 1)
])

dataloader = torch.utils.data.DataLoader(
    datasets.MNIST('./data', train=True, download=True, transform=transform),
    batch_size=batch_size,
    shuffle=True,
    drop_last=True,
)

## 3. Weight Initialization

The DCGAN paper recommends:

- Conv / ConvTranspose weights: $\mathcal{N}(0, 0.02)$
- BatchNorm weights: $\mathcal{N}(1, 0.02)$
- BatchNorm biases: constant $0$

This is a small detail that has a real effect on training stability. Without it, activations can drift out of reasonable ranges in the first few iterations.

In [4]:
def weights_init(m):
    """DCGAN-style weight initialization."""
    classname = m.__class__.__name__
    if classname.find('Conv') != -1:
        nn.init.normal_(m.weight.data, 0.0, 0.02)
    elif classname.find('BatchNorm') != -1:
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0)

## 4. The Generator

The Generator starts from a latent vector $z \in \mathbb{R}^{100}$ and progressively upsamples to a 64×64 image.

Each block is **ConvTranspose2d → BatchNorm2d → ReLU**. The final block ends with `Tanh` so outputs lie in $[-1, +1]$.

Spatial flow:

```
1 × 1   →  4 × 4   →  8 × 8   →  16 × 16  →  32 × 32  →  64 × 64
  100     512       256         128          64          1
```

So the Generator goes:

$$z \in \mathbb{R}^{100} \to \mathbb{R}^{1 \times 64 \times 64}$$

In [5]:
class Generator(nn.Module):
    def __init__(self, latent_dim=100, img_channels=1, features_g=64):
        super().__init__()
        self.net = nn.Sequential(
            # z: (B, latent_dim, 1, 1) -> (B, features_g*8, 4, 4)
            nn.ConvTranspose2d(latent_dim, features_g * 8, kernel_size=4, stride=1, padding=0, bias=False),
            nn.BatchNorm2d(features_g * 8),
            nn.ReLU(True),

            # 4 -> 8
            nn.ConvTranspose2d(features_g * 8, features_g * 4, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(features_g * 4),
            nn.ReLU(True),

            # 8 -> 16
            nn.ConvTranspose2d(features_g * 4, features_g * 2, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(features_g * 2),
            nn.ReLU(True),

            # 16 -> 32
            nn.ConvTranspose2d(features_g * 2, features_g, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(features_g),
            nn.ReLU(True),

            # 32 -> 64, project to img_channels, Tanh
            nn.ConvTranspose2d(features_g, img_channels, kernel_size=4, stride=2, padding=1, bias=False),
            nn.Tanh(),
        )

    def forward(self, z):
        # z is (B, latent_dim) — reshape to (B, latent_dim, 1, 1)
        return self.net(z.view(z.size(0), z.size(1), 1, 1))

## 5. The Discriminator

The Discriminator is roughly the Generator mirrored. Each block is **Conv2d (stride=2) → BatchNorm2d → LeakyReLU**, and we **do not** use a `Sigmoid` at the end.

Why no Sigmoid? We use `BCEWithLogitsLoss`, which is numerically more stable than applying `Sigmoid` + `BCELoss` separately. The Discriminator outputs a **logit**, not a probability.

Spatial flow:

```
64 × 64  →  32 × 32  →  16 × 16  →  8 × 8   →  4 × 4   →  1
   1         64          128         256        512       1
```

Notice: no BatchNorm in the **first** Conv layer of the Discriminator. This is a DCGAN-paper recommendation — it prevents the discriminator from immediately leaking information about the input distribution through BatchNorm statistics.

In [6]:
class Discriminator(nn.Module):
    def __init__(self, img_channels=1, features_d=64):
        super().__init__()
        self.net = nn.Sequential(
            # 64 -> 32, NO BatchNorm in the first block
            nn.Conv2d(img_channels, features_d, kernel_size=4, stride=2, padding=1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),

            # 32 -> 16
            nn.Conv2d(features_d, features_d * 2, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(features_d * 2),
            nn.LeakyReLU(0.2, inplace=True),

            # 16 -> 8
            nn.Conv2d(features_d * 2, features_d * 4, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(features_d * 4),
            nn.LeakyReLU(0.2, inplace=True),

            # 8 -> 4
            nn.Conv2d(features_d * 4, features_d * 8, kernel_size=4, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(features_d * 8),
            nn.LeakyReLU(0.2, inplace=True),

            # 4 -> 1 (logit)
            nn.Conv2d(features_d * 8, 1, kernel_size=4, stride=1, padding=0, bias=False),
            # No Sigmoid — we use BCEWithLogitsLoss
        )

    def forward(self, x):
        return self.net(x).view(x.size(0), -1)

## 6. Models, Optimizers, and Loss

Loss: `BCEWithLogitsLoss` (numerically stable — combines `Sigmoid` + `BCELoss` in one shot).

Two Adam optimizers, separate LR for `G` and `D` is fine but here we use the same `2e-4`. The DCGAN paper used the same LR for both.

In [7]:
G = Generator(latent_dim=latent_dim, img_channels=img_channels, features_g=features_g).to(device)
D = Discriminator(img_channels=img_channels, features_d=features_d).to(device)

G.apply(weights_init)
D.apply(weights_init)

optimizer_G = optim.Adam(G.parameters(), lr=lr, betas=betas)
optimizer_D = optim.Adam(D.parameters(), lr=lr, betas=betas)

criterion = nn.BCEWithLogitsLoss()

print(G)
print()
print(D)

Generator(
  (net): Sequential(
    (0): ConvTranspose2d(100, 512, kernel_size=(4, 4), stride=(1, 1), bias=False)
    (1): BatchNorm2d(512, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): ConvTranspose2d(512, 256, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1), bias=False)
    (4): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (5): ReLU(inplace=True)
    (6): ConvTranspose2d(256, 128, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1), bias=False)
    (7): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (8): ReLU(inplace=True)
    (9): ConvTranspose2d(128, 64, kernel_size=(4, 4), stride=(2, 2), padding=(1, 1), bias=False)
    (10): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
    (11): ReLU(inplace=True)
    (12): ConvTranspose2d(64, 1, kernel_size=(4, 4), stride=(2, 2), padding=

## 7. The Training Loop

Same alternating structure as the vanilla GAN (Module 5), just with convolutional networks:

**Phase A — Train D**

1. Real images → D → logit
2. Fake images (detached) → D → logit
3. BCE loss with targets `1` and `0`
4. Backprop into D only

**Phase B — Train G**

1. Generate fakes (no detach)
2. D(fakes) → logit
3. BCE loss with target `1` (non-saturating, same as Module 5)
4. Backprop through D into G, update only G

In [ ]:
fixed_noise = torch.randn(64, latent_dim, 1, 1, device=device)

losses_g = []
losses_d = []

G.train()
D.train()

for epoch in range(epochs):
    d_running_loss = 0.0
    g_running_loss = 0.0
    n_batches = 0

    for real_imgs, _ in dataloader:
        real_imgs = real_imgs.to(device)
        b = real_imgs.size(0)

        valid = torch.ones(b, 1, device=device)
        fake   = torch.zeros(b, 1, device=device)

        # -----------------
        # Phase A: Train D
        # -----------------
        optimizer_D.zero_grad()

        # Real images -> label 1
        real_logits = D(real_imgs)
        d_real_loss = criterion(real_logits, valid)

        # Fake images -> label 0, detached
        z = torch.randn(b, latent_dim, 1, 1, device=device)
        with torch.no_grad():
            gen_imgs = G(z)
        fake_logits = D(gen_imgs)
        d_fake_loss = criterion(fake_logits, fake)

        d_loss = (d_real_loss + d_fake_loss) / 2
        d_loss.backward()
        optimizer_D.step()

        # -----------------
        # Phase B: Train G
        # -----------------
        optimizer_G.zero_grad()

        gen_imgs = G(z)                  # NO detach — gradient flows through D into G
        validity_logits = D(gen_imgs)
        g_loss = criterion(validity_logits, valid)   # want D to label our fakes as real

        g_loss.backward()
        optimizer_G.step()

        d_running_loss += d_loss.item()
        g_running_loss += g_loss.item()
        n_batches += 1

    avg_d = d_running_loss / n_batches
    avg_g = g_running_loss / n_batches
    losses_d.append(avg_d)
    losses_g.append(avg_g)

    print(f"Epoch [{epoch+1}/{epochs}]  D: {avg_d:.4f}  G: {avg_g:.4f}")

    # Save a sample grid every epoch
    G.eval()
    with torch.no_grad():
        sample_imgs = G(fixed_noise).detach().cpu()
    save_image(sample_imgs, f"samples/epoch_{epoch+1:02d}.png", nrow=8, normalize=True)
    G.train()

print('Training done.')

Epoch [1/25]  D: 0.2883  G: 4.4816
Epoch [2/25]  D: 0.2633  G: 3.1025
Epoch [3/25]  D: 0.2370  G: 3.6862
Epoch [4/25]  D: 0.2158  G: 3.8297


## 8. Generating New Samples

After training, sample from the Generator by feeding it random noise.

In [ ]:
G.eval()
with torch.no_grad():
    samples = G(torch.randn(64, latent_dim, 1, 1, device=device)).cpu()

grid = make_grid(samples, nrow=8, normalize=True)
plt.figure(figsize=(8, 8))
plt.imshow(grid.permute(1, 2, 0).squeeze(), cmap='gray')
plt.axis('off')
plt.title('Generated MNIST samples (DCGAN)')
plt.show()

## 9. Training Progression

Read each saved grid in order to see how the Generator's output evolves from noise to recognizable digits.

In [ ]:
from PIL import Image
import glob

paths = sorted(glob.glob('samples/epoch_*.png'))
if paths:
    fig, axes = plt.subplots(1, len(paths), figsize=(2.2 * len(paths), 2.2))
    if len(paths) == 1:
        axes = [axes]
    for ax, p in zip(axes, paths):
        ax.imshow(np.array(Image.open(p)).squeeze(), cmap='gray')
        ax.set_title(p.split('_')[-1].split('.')[0])
        ax.axis('off')
    plt.suptitle('Generator progression across epochs')
    plt.show()
else:
    print('No sample grids found. Run the training cell first.')

## 10. Loss Curves

DCGAN is more stable than vanilla GAN, but you'll still see oscillation. The shapes of the curves are easier to interpret than for vanilla GAN because BatchNorm keeps activations in check.

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(losses_d, label='Discriminator')
plt.plot(losses_g, label='Generator')
plt.xlabel('Epoch')
plt.ylabel('BCELoss')
plt.title('DCGAN losses')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

## Recap

What changed from `vanillaGAN.ipynb` to here:

| Concern | Vanilla GAN | DCGAN |
|---|---|---|
| Generator body | 4 fully-connected layers | `ConvTranspose2d` blocks |
| Discriminator body | 4 fully-connected layers | strided `Conv2d` blocks |
| Activation (G) | LeakyReLU | **ReLU** + final Tanh |
| Activation (D) | LeakyReLU + Sigmoid | **LeakyReLU** + **logit** |
| Loss | `BCELoss` | **`BCEWithLogitsLoss`** |
| Normalization | Dropout (D only) | **BatchNorm** (G & D, except D's first layer) |
| Weight init | PyTorch defaults | **DCGAN-style** N(0, 0.02) |
| Spatial structure | destroyed by flatten | **preserved throughout** |

Same training loop, same minimax game, same `detach()` discipline. Only the **architecture** changed — and that alone gives a much stronger image model.

**Next in the series** (Module 10): the engineering tricks that take us from "DCGAN mostly works" to "DCGAN trains reliably" — label smoothing, learning-rate tuning, and more.